# Pastry Sales Forecasting — Data Preparation

This notebook uses the public synthetic dataset by default. Locally, the same pipeline can use confidential work data when `PASTRY_DATA_MODE=private`. The private files and configuration are excluded by `.gitignore`.

> Before committing this notebook, run it again in synthetic mode and verify that no confidential outputs remain.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.data_preparation as dp
from src.create_synthetic_raw_files import create_synthetic_raw_files

pd.set_option("display.max_rows", 100)

## Select synthetic or confidential local data


In [ ]:
import os

DATA_MODE = os.environ.get("PASTRY_DATA_MODE", "synthetic").strip().lower()
if DATA_MODE not in {"synthetic", "private"}:
    raise ValueError("PASTRY_DATA_MODE must be either synthetic or private")

if DATA_MODE == "private":
    private_config = PROJECT_ROOT / "private_config" / "data_config.py"
    if not private_config.exists():
        raise FileNotFoundError(
            "Private mode was requested, but private_config/data_config.py was not found."
        )

    from private_config.data_config import (
        PRODUCTION_FILENAME,
        STORE_FILES,
        STORES,
    )

    RAW_DATA_PATH = PROJECT_ROOT / "data" / "private"
    PROCESSED_DATA_PATH = RAW_DATA_PATH / "processed"
    OUTPUTS_PATH = PROJECT_ROOT / "outputs" / "private"
    store_paths = STORE_FILES
    production_file = RAW_DATA_PATH / PRODUCTION_FILENAME
    stores = STORES
else:
    RAW_DATA_PATH = PROJECT_ROOT / "data" / "synthetic_raw"
    PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"
    OUTPUTS_PATH = PROJECT_ROOT / "outputs"
    store_paths = {
        "Shop_A": "Shop_A_losses.xlsx",
        "Shop_B": "Shop_B_losses.xlsx",
        "Shop_C": "Shop_C_losses.xlsx",
    }
    production_file = RAW_DATA_PATH / "synthetic_production.xlsx"
    stores = list(store_paths)

for folder in [RAW_DATA_PATH, PROCESSED_DATA_PATH, OUTPUTS_PATH]:
    folder.mkdir(parents=True, exist_ok=True)

required_files = [
    production_file,
    *(RAW_DATA_PATH / name for name in store_paths.values()),
]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required data files:\n"
        + "\n".join(str(path) for path in missing_files)
    )

print("Data mode:", DATA_MODE)
print("Data folder:", RAW_DATA_PATH)


The synthetic files are included in the repository. Run the next cell only when you want to regenerate them. It is intentionally disabled in private mode.


In [ ]:
if DATA_MODE == "synthetic":
    # create_synthetic_raw_files(
    #     output_dir=RAW_DATA_PATH,
    #     start_date="2026-05-01",
    #     end_date="2026-07-31",
    #     seed=42,
    #     include_anomalies=False,
    # )
    pass


## Load production workbooks

The raw Japanese columns are converted to English by `load_production()`:

- `残在庫` → `ClosingStock`
- `製造数` → `Production`
- `予測` is removed because it is not used by the demand model.

`ClosingStock` remains on the workbook date. In private mode, every recorded `Production` value is aligned to the following selling date. `CarryoverStock` is then calculated from the previous day's `ClosingStock`.


In [ ]:
excel = pd.ExcelFile(production_file)
print(excel.sheet_names)

# Use up to twelve monthly sheets after the response/form template sheets.
sheet_names = excel.sheet_names[2:14]


In [ ]:
production = dp.load_production(
    production_file,
    stores,
    sheet_names,
)

# The production written under a workbook date is sold the following day.
# ClosingStock stays on the date when the remaining stock was counted.
if DATA_MODE == "private":
    production = dp.align_production_to_sales_date(production)

private_business_rules = (
    PROJECT_ROOT
    / "private_config"
    / "business_rules.py"
)

if DATA_MODE == "private" and private_business_rules.exists():
    from private_config.business_rules import (
        apply_known_business_rules,
    )

    production = apply_known_business_rules(production)
else:
    production["ShopOpen"] = True
    production["ClosedReason"] = pd.NA

display(production.head(10))


In [ ]:
if DATA_MODE == "private":
    display(
        production[
            [
                "Date",
                "Store",
                "Product",
                "ClosingStock",
                "Production",
            ]
        ]
        .sort_values(["Date", "Store", "Product"])
        .tail(20)
    )


## Add Excel sales and loss, then inventory sales

The first function reads loss and available sales from the shop forms in one pass. `客数` is ignored. The second function calculates sales from inventory, fills missing sales, and compares the two sources.


In [ ]:
production = dp.add_product_type(production)

# Public mode uses fictional demonstration prices and set definitions.
# Private mode loads the real rules from a local, Git-ignored module.
if DATA_MODE == "private":
    private_sales_rules = (
        PROJECT_ROOT
        / "private_config"
        / "sales_rules.py"
    )
    if not private_sales_rules.exists():
        raise FileNotFoundError(
            "Private mode requires private_config/sales_rules.py."
        )

    from private_config.sales_rules import SALES_RULES
else:
    SALES_RULES = dp.DEMO_SALES_RULES

production = dp.add_excel_sales_and_loss(
    production,
    store_paths,
    RAW_DATA_PATH,
    sales_rules=SALES_RULES,
)

production = dp.add_inventory_sales(production)
production = dp.add_stock_age(production)
production = dp.mark_usable_rows(production)

# Add the supplied product economics without making a tax assumption.
# These values are used for later profit and waste evaluation only.
production["ListedPrice"] = pd.NA
production["UnitCost"] = pd.NA
production["UnitMarginEstimate"] = pd.NA
production["EconomicsSource"] = pd.NA

private_economics_config = (
    PROJECT_ROOT
    / "private_config"
    / "product_economics.py"
)

if DATA_MODE == "private" and private_economics_config.exists():
    from private_config.product_economics import (
        PRODUCT_ECONOMICS,
        SEASONAL_ECONOMICS,
    )

    price_map = {
        product: values["ListedPrice"]
        for product, values in PRODUCT_ECONOMICS.items()
    }
    cost_map = {
        product: values["UnitCost"]
        for product, values in PRODUCT_ECONOMICS.items()
    }

    production["ListedPrice"] = production["Product"].map(price_map)
    production["UnitCost"] = production["Product"].map(cost_map)
    production.loc[
        production["ListedPrice"].notna(),
        "EconomicsSource",
    ] = "product"

    use_seasonal_default = (
        production["Type"].eq("seasonal")
        & production["ListedPrice"].isna()
    )
    production.loc[
        use_seasonal_default,
        "ListedPrice",
    ] = SEASONAL_ECONOMICS["ListedPrice"]
    production.loc[
        use_seasonal_default,
        "UnitCost",
    ] = SEASONAL_ECONOMICS["UnitCost"]
    production.loc[
        use_seasonal_default,
        "EconomicsSource",
    ] = "seasonal_default"

    production["ListedPrice"] = pd.to_numeric(
        production["ListedPrice"],
        errors="coerce",
    )
    production["UnitCost"] = pd.to_numeric(
        production["UnitCost"],
        errors="coerce",
    )
    production["UnitMarginEstimate"] = (
        production["ListedPrice"]
        - production["UnitCost"]
    )
else:
    print("No private product-economics configuration was loaded.")

production = production.sort_values(
    ["Date", "Store", "Product"]
).reset_index(drop=True)

display(production.head(30))


### Check known promotion events

In private mode, display rows where the local business-rule configuration adds a promotion or event feature. Synthetic mode contains no real company event details.

In [ ]:
event_columns = [
    "Date",
    "Store",
    "Product",
    "IsEventDay",
    "EventName",
    "PromotionType",
    "PromotionTarget",
    "DiscountRate",
    "FreeChouxPerCustomer",
    "MaxPromotionCustomers",
    "MinimumPurchaseUnits",
    "AnyProductQualifies",
]

# Promotion columns exist only when the confidential business-rule module is
# loaded. Synthetic mode intentionally contains no real company event details.
if "IsEventDay" in production.columns:
    display(
        production.loc[production["IsEventDay"], event_columns]
        .sort_values(["Date", "Store", "Product"])
    )
else:
    print("No private promotion events are included in synthetic mode.")


In [ ]:
diagnostic_columns = [
    "Date",
    "Store",
    "Product",
    "CarryoverStock",
    "Production",
    "Loss",
    "ClosingStock",
    "ExcelSales",
    "InventorySales",
    "Sales",
    "SalesSource",
    "Diagnostics",
]

print("Rows with diagnostics")
display(
    production.loc[
        production["Diagnostics"].notna(),
        diagnostic_columns,
    ].sort_values(["Date", "Store", "Product"])
)


In [ ]:
duplicate_count = production.duplicated(
    ["Date", "Store", "Product"]
).sum()

print("Rows:", len(production))
print("Duplicate keys:", duplicate_count)
print(
    "Rows with diagnostics:",
    int(production["Diagnostics"].notna().sum()),
)
print(
    "Rows usable for sales model:",
    int(production["UseForSalesModel"].fillna(False).sum()),
)

print("\nSales source")
display(
    production["SalesSource"]
    .value_counts(dropna=False)
    .to_frame("Rows")
)

model_rows = production.loc[
    production["UseForSalesModel"].fillna(False)
]

assert duplicate_count == 0
assert model_rows["Sales"].notna().all()
assert not model_rows["Sales"].lt(0).any()

print("\nModel-target validation passed.")


## Calendar and historical weather

Create weekday, Japanese holiday, and historical weather features for the dates already present in the Excel data.


In [ ]:
from src.calendar_features import (
    create_calendar,
    add_historical_weather,
)

calendar = create_calendar(production)

if DATA_MODE == "private":
    from private_config.calendar_config import (
        WEATHER_LATITUDE,
        WEATHER_LONGITUDE,
        WEATHER_TIMEZONE,
    )

    calendar = add_historical_weather(
        calendar,
        latitude=WEATHER_LATITUDE,
        longitude=WEATHER_LONGITUDE,
        timezone=WEATHER_TIMEZONE,
    )
else:
    for column in [
        "Weather",
        "TemperatureMin",
        "TemperatureMax",
        "TemperatureAvg",
    ]:
        calendar[column] = pd.NA

display(calendar.head())


In [ ]:
calendar_columns = [
    "Weekday",
    "DayType",
    "Weather",
    "TemperatureMin",
    "TemperatureMax",
    "TemperatureAvg",
]

production = production.drop(
    columns=[
        column
        for column in calendar_columns
        if column in production.columns
    ],
    errors="ignore",
)

production = production.merge(
    calendar,
    on="Date",
    how="left",
    validate="many_to_one",
)


## Keep the model period and save

After all normal preparation is complete, keep only **2026-05-22 through 2026-07-31** for the model dataset.


In [ ]:
MODEL_START_DATE = pd.Timestamp("2026-05-22")
MODEL_END_DATE = pd.Timestamp("2026-07-31")

model_data = production.loc[
    production["Date"].between(
        MODEL_START_DATE,
        MODEL_END_DATE,
    )
].copy()

model_data = model_data.sort_values(
    ["Date", "Store", "Product"]
).reset_index(drop=True)

if model_data.empty:
    raise ValueError("No model rows were found in the selected date range.")

PICKLE_PATH = PROCESSED_DATA_PATH / "production.pkl"
EVALUATION_PATH = PROCESSED_DATA_PATH / "production_evaluation.pkl"
EXCEL_PATH = OUTPUTS_PATH / "production_cleaning.xlsx"

# The training dataset stops on July 31. The separate evaluation file keeps
# later prepared rows so a completed forecast can be compared with actuals
# without exposing those actuals to model training.
model_data.to_pickle(PICKLE_PATH)
production.to_pickle(EVALUATION_PATH)
model_data.to_excel(EXCEL_PATH, index=False)

print("Model data period:", model_data["Date"].min(), "to", model_data["Date"].max())
print("Model data rows:", len(model_data))
print("Saved model data:", PICKLE_PATH)
print("Saved evaluation data:", EVALUATION_PATH)
print("Saved cleaning review:", EXCEL_PATH)